# Power LLaVA-NeXT's has_error=1 stratum

Eleventh notebook. LLaVA-NeXT's n=300 reference run (notebook 10) found a
capability gate on the perception arm (0.8% accuracy on free-response
items -- not testable, see `report`/CLAUDE.md), but the reasoning arm looks
genuinely promising and is unaffected by that gate: grading doesn't need
fine-grained OCR, just a holistic judgment. The clean-stratum point
estimate (0.283) is almost identical to Qwen-7B's confirmed 0.280, but
both LLaVA strata are underpowered at n=300 (14 misgraded each, need 30+).

This notebook mirrors notebooks 06/08/09 exactly: draw more
`has_error=1` items, grading-only (no need to redo transcription -- the
perception arm on this model is a separate, already-diagnosed question),
merge with the reference run, and check whether the reasoning-arm
replication holds up with real power.

## Sizing

LLaVA's `has_error=1` error rate is 14/150 = 9.3% (90% Wilson CI
[6.1%, 14.0%]) -- between 3B's 5.3% and 7B's 11.3%. Planning off the point
estimate (matching what actually worked for both prior extensions,
\S7.4/\S7.5 of the report): **N_EXTRA=250** gives an expected total of
~37 misgraded items, a comfortable margin over the registered minimum of
30, similar in proportion to 7B's 200-item extension (which landed at 38).

**Prerequisite for running:** cells 2-3 (install, auth) must run this
session. Model load is LLaVA-NeXT, same as notebook 10.


In [ ]:
# Install cell: GPU-dependent packages only.
%pip install -q transformers accelerate datasets huggingface_hub bitsandbytes


In [ ]:
# Auth & code/results access cell. Identical to notebooks 06/09/10.
import json
import os
from getpass import getpass

from huggingface_hub import login

from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
DRIVE_MODEL_CACHE = f"{PROJECT_DIR}/model_cache"
os.makedirs(DRIVE_MODEL_CACHE, exist_ok=True)

TOKEN_FILE = f"{PROJECT_DIR}/.tokens.json"
RESET_TOKENS = False


def get_token(name, prompt):
    tokens = {}
    if os.path.exists(TOKEN_FILE):
        with open(TOKEN_FILE) as f:
            tokens = json.load(f)
    if RESET_TOKENS or not tokens.get(name):
        tokens[name] = getpass(prompt).strip()
        with open(TOKEN_FILE, "w") as f:
            json.dump(tokens, f)
        os.chmod(TOKEN_FILE, 0o600)
        print(f"Saved {name} to Drive -- you will not be asked for it again.")
    return tokens[name]


HF_TOKEN = get_token("HF_TOKEN", "Hugging Face token (asked once): ")
GH_TOKEN = get_token("GH_TOKEN", "GitHub token with 'repo' scope (asked once): ")

if not HF_TOKEN.startswith("hf_"):
    raise ValueError(
        "Stored Hugging Face token does not start with 'hf_'. Set "
        "RESET_TOKENS = True and re-run this cell to replace it."
    )

login(token=HF_TOKEN)
print("Hugging Face login OK")

REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"
!rm -rf repo
!git clone -q {REPO_URL} repo
%pip install -q -e repo/

import importlib
import sys

REPO_DIR = os.path.abspath("repo")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()

import pilot.data
import pilot.prompts
import pilot.parsing
import pilot.entropy
import pilot.plotting

print(f"pilot package imported from: {os.path.dirname(pilot.__file__)}")


In [ ]:
# Model load cell. Identical to notebook 10.
import torch
from transformers import AutoProcessor, LlavaNextForConditionalGeneration

MODEL_ID = "llava-hf/llava-v1.6-mistral-7b-hf"
QUANTIZED = False

try:
    model = LlavaNextForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        cache_dir=DRIVE_MODEL_CACHE,
    )
    print(f"Loaded {MODEL_ID} in bfloat16 (full precision).")
except torch.cuda.OutOfMemoryError:
    print(f"bfloat16 load of {MODEL_ID} did not fit -- falling back to 4-bit "
          "quantization. This changes what is being measured; the saved "
          "results record QUANTIZED=True so this is never silently glossed over.")
    from transformers import BitsAndBytesConfig

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = LlavaNextForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        device_map="auto",
        cache_dir=DRIVE_MODEL_CACHE,
    )
    QUANTIZED = True

processor = AutoProcessor.from_pretrained(MODEL_ID, cache_dir=DRIVE_MODEL_CACHE)
processor.patch_size = model.config.vision_config.patch_size
processor.vision_feature_select_strategy = model.config.vision_feature_select_strategy
processor.num_additional_image_tokens = 1
processor.tokenizer.padding_side = "left"

if torch.cuda.is_available():
    vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {torch.cuda.get_device_name(0)} ({vram_gib:.1f} GiB), "
          f"quantized={QUANTIZED}")


In [ ]:
# Extra-items sample cell. Draws ONLY new has_error=1 items, disjoint from
# the 150 the reference run (notebook 10) already used.
import logging

import pilot.data

logging.basicConfig(level=logging.WARNING, force=True)

SEED = 42
SKIP = 150          # has_error=1 items the reference run already covered
N_EXTRA = 250        # see the intro cell for the sizing rationale

extra_sample = pilot.data.load_fermat_extra_error_items(
    n_extra=N_EXTRA, seed=SEED, skip=SKIP
)
N_EXTRA = len(extra_sample)  # may shrink if the pool ran short -- keep in sync
print(f"{N_EXTRA} additional has_error=1 items drawn (items {SKIP} to {SKIP + N_EXTRA - 1} "
      f"in the seed={SEED} error-item ordering, disjoint from the reference run's first {SKIP})")


In [ ]:
# LLaVA-NeXT adapter, grading only -- identical message-building approach
# as notebook 10 (no system role, folded into one user turn; combined
# apply_chat_template call). No pre-flight cell here since the adapter is
# already verified working end-to-end by notebook 10's real run.
import pilot.prompts


def build_llava_messages(system_prompt: str, user_prompt: str, image) -> list[dict]:
    combined_text = f"{system_prompt}\n\n{user_prompt}"
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": combined_text},
            ],
        },
    ]


def build_llava_grading_messages(image):
    return build_llava_messages(
        pilot.prompts.GRADING_SYSTEM_PROMPT, pilot.prompts.GRADING_USER_PROMPT, image
    )


In [ ]:
# Grading generation for the extra items. K=5, same batch-backoff ladder as
# every prior grading notebook. Checkpoint entries use grading_samples_raw
# (matching the reference run's key name from notebook 10) so the merge
# cell can use one score_entry function for both checkpoints.
import gc
import json
import os
import time

from tqdm.auto import tqdm

K_GRADING = 5
TEMP = 0.7
_BATCH_LADDER = [5, 2, 1]
_batch_state = {"index": 0}

META_FIELDS = ("orig_q", "pert_a", "has_error", "handwriting_style", "image_quality")
INFRA_EXCEPTIONS = (ConnectionError, TimeoutError, torch.cuda.OutOfMemoryError, OSError)


def _generate_batch(messages, n: int, temperature: float):
    inputs = processor.apply_chat_template(
        messages, tokenize=True, return_dict=True,
        return_tensors="pt", add_generation_prompt=True,
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=512, do_sample=True,
            temperature=temperature, num_return_sequences=n,
        )

    trimmed = output_ids[:, inputs["input_ids"].shape[1]:]
    texts = processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    del output_ids, inputs
    gc.collect()
    torch.cuda.empty_cache()
    return texts


def generate_grading(messages, n: int, temperature: float):
    texts = []
    last_exc = None
    while len(texts) < n:
        want = n - len(texts)
        size = min(_BATCH_LADDER[_batch_state["index"]], want)
        for attempt in range(3):
            try:
                texts += _generate_batch(messages, size, temperature)
                last_exc = None
                break
            except torch.cuda.OutOfMemoryError:
                gc.collect()
                torch.cuda.empty_cache()
                if _batch_state["index"] + 1 < len(_BATCH_LADDER):
                    _batch_state["index"] += 1
                    print(f"  OOM at batch {size}; dropping to "
                          f"{_BATCH_LADDER[_batch_state['index']]} for the rest of the run.",
                          flush=True)
                    size = min(_BATCH_LADDER[_batch_state["index"]], n - len(texts))
                    continue
                raise
            except INFRA_EXCEPTIONS as exc:
                last_exc = exc
                gc.collect()
                torch.cuda.empty_cache()
                if attempt < 2:
                    time.sleep(5)
        if last_exc is not None:
            raise last_exc
    return texts


CHECKPOINT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
model_slug = MODEL_ID.split("/")[-1]
extra_grading_path = (f"{CHECKPOINT_DIR}/grading_llava_extra_error_k{K_GRADING}_{model_slug}"
                      f"_n{N_EXTRA}_skip{SKIP}_seed{SEED}{'_4bit' if QUANTIZED else ''}.jsonl")

extra_grading_results = []
if os.path.exists(extra_grading_path):
    with open(extra_grading_path) as f:
        extra_grading_results = [json.loads(line) for line in f if line.strip()]
    valid = []
    for idx, entry in enumerate(extra_grading_results[:N_EXTRA]):
        item = extra_sample[idx]
        if not all(entry["item"].get(k) == item[k] for k in META_FIELDS):
            print(f"Checkpoint item {idx + 1} does not match sample order; resuming there.")
            break
        if len(entry.get("grading_samples_raw", [])) != K_GRADING:
            break
        valid.append(entry)
    if len(valid) != len(extra_grading_results):
        with open(extra_grading_path, "w") as f:
            for e in valid:
                f.write(json.dumps(e, default=str) + "\n")
        print(f"Truncated checkpoint from {len(extra_grading_results)} to {len(valid)} valid items.")
    extra_grading_results = valid
    print(f"Resuming from {len(extra_grading_results)} completed items")

if len(extra_grading_results) >= N_EXTRA:
    print(f"All {N_EXTRA} items already done.")
else:
    print(f"Starting from item {len(extra_grading_results) + 1}/{N_EXTRA} "
          f"({N_EXTRA - len(extra_grading_results)} remaining)", flush=True)
    with tqdm(total=(N_EXTRA - len(extra_grading_results)) * K_GRADING,
              desc="grading (LLaVA extra)", unit="sample") as pbar:
        for item_idx, item in enumerate(extra_sample):
            if item_idx < len(extra_grading_results):
                continue
            _t0 = time.time()
            messages = build_llava_grading_messages(item["image"])
            texts = generate_grading(messages, K_GRADING, TEMP)
            _elapsed = time.time() - _t0
            entry = {
                "item": {k: item[k] for k in META_FIELDS},
                "grading_samples_raw": texts,
                "quantized": QUANTIZED,
                "elapsed_seconds": _elapsed,
            }
            extra_grading_results.append(entry)
            with open(extra_grading_path, "a") as f:
                f.write(json.dumps(entry, default=str) + "\n")
                f.flush()
            pbar.update(K_GRADING)
            print(f"  item {item_idx + 1}/{N_EXTRA}: {_elapsed:.1f}s "
                  f"({len(extra_grading_results)}/{N_EXTRA} done)", flush=True)

print(f"extra_grading_results: {len(extra_grading_results)} items")


In [ ]:
# Merge with the LLaVA reference run (notebook 10) and re-run the
# stratified analysis. Reads the REFERENCE CHECKPOINT (raw text) directly.
#
# Schema note: the reference checkpoint (notebook 10) has BOTH
# transcription_samples_raw and grading_samples_raw (this notebook only
# needs the latter) and DOES have a quantized field (unlike the 3B
# reference from notebook 03) -- so the precision-consistency check here
# mirrors notebook 06/09's pattern, not notebook 08's model_id-only check.
import importlib
import json
import os

import pandas as pd

import pilot.entropy
import pilot.parsing
import pilot.plotting

for m in (pilot.parsing, pilot.entropy, pilot.plotting):
    importlib.reload(m)


def _find_checkpoint(base, label):
    candidates = {"bf16": f"{base}.jsonl", "4bit": f"{base}_4bit.jsonl"}
    found = {k: p for k, p in candidates.items() if os.path.exists(p)}
    if not found:
        raise AssertionError(
            f"No {label} checkpoint found at {candidates['bf16']} or {candidates['4bit']}."
        )
    if len(found) > 1:
        raise AssertionError(
            f"Found {label} checkpoints under BOTH precisions: {list(found)}. "
            "Resolve manually before merging."
        )
    return next(iter(found.values()))


REFERENCE_CHECKPOINT = _find_checkpoint(
    f"{CHECKPOINT_DIR}/scaleup_{model_slug}_n300_seed{SEED}_bal50_kt5_kg5",
    "reference",
)

with open(REFERENCE_CHECKPOINT) as f:
    reference_entries = [json.loads(line) for line in f if line.strip()]
assert len(reference_entries) == 300, (
    f"Expected 300 reference items, found {len(reference_entries)}."
)

# Precision consistency across BOTH checkpoints, every entry, not just [0].
def _quantized_values(entries):
    return {bool(e["quantized"]) for e in entries}


ref_quantized = _quantized_values(reference_entries)
extra_quantized = _quantized_values(extra_grading_results)
all_quantized = ref_quantized | extra_quantized
if all_quantized != {QUANTIZED}:
    raise RuntimeError(
        f"Quantization mismatch: reference={ref_quantized}, extra={extra_quantized}, "
        f"this session={QUANTIZED}. Refusing to merge measurements taken under "
        "different precisions."
    )
print(f"Precision check OK: both runs quantized={QUANTIZED}")


def score_entry(entry):
    digits = [pilot.parsing.parse_grading(t) for t in entry["grading_samples_raw"]]
    labels = [None if d is None else str(d) for d in digits]
    majority, _ = pilot.entropy.majority_cluster(labels)
    said_error = majority == "1"
    return {
        "orig_q": entry["item"]["orig_q"],
        "pert_a": entry["item"]["pert_a"],
        "has_error": entry["item"]["has_error"],
        "grading_correct": majority in {"0", "1"} and said_error == bool(entry["item"]["has_error"]),
        "said_error": said_error,
        "reasoning_entropy": pilot.entropy.cluster_entropy(labels),
        "n_grading_parse_failures": sum(1 for d in digits if d is None),
        "majority_digit": majority,
        "parsed_digits": digits,
        "all_grading_samples_raw": entry["grading_samples_raw"],
        "model_id": MODEL_ID,
        "quantized": entry["quantized"],
    }


reference_df = pd.DataFrame(score_entry(e) for e in reference_entries)
extra_df = pd.DataFrame(score_entry(e) for e in extra_grading_results)

overlap = set(zip(reference_df["orig_q"], reference_df["pert_a"])) & \
          set(zip(extra_df["orig_q"], extra_df["pert_a"]))
assert not overlap, f"{len(overlap)} items overlap between reference and extra -- sampling bug"

combined = pd.concat([reference_df, extra_df], ignore_index=True)
print(f"Combined: {len(reference_df)} reference + {len(extra_df)} extra = {len(combined)} total")

gt = combined["has_error"].astype(bool)
print(f"  has_error=1: {int(gt.sum())} items, {int((~combined.loc[gt,'grading_correct']).sum())} misgraded")
print(f"  has_error=0: {int((~gt).sum())} items, {int((~combined.loc[~gt,'grading_correct']).sum())} misgraded")

print()
print("=" * 70)
print("STRATIFIED ANALYSIS (combined, LLaVA-NeXT)")
print("=" * 70)
out = pilot.plotting.stratified_auroc(
    combined, "reasoning_entropy", "grading_correct", "has_error", n_boot=10000, seed=0
)
for level, s in out["strata"].items():
    minority = min(s["n_error"], s["n_correct"])
    powered = minority >= pilot.plotting.SCALEUP_PREREGISTRATION["min_minority_class"]
    print(f"  has_error={level}  n={s['n_items']:3d}  n_wrong={s['n_error']:3d}  "
          f"AUROC {s['auroc']:.3f} [{s['ci_low']:.3f}, {s['ci_high']:.3f}]  "
          f"minority={minority}  {'POWERED' if powered else 'still underpowered'}")
print(f"  sign_reversal      : {out['sign_reversal']}")
print(f"  pooled_understates : {out['pooled_understates']}")

print()
print("=" * 70)
print("VERDICT (reusing the registered 0.70 threshold, not a new one)")
print("=" * 70)
error_stratum = out["strata"][True]
minority = min(error_stratum["n_error"], error_stratum["n_correct"])
min_n = pilot.plotting.SCALEUP_PREREGISTRATION["min_minority_class"]
threshold = pilot.plotting.SCALEUP_PREREGISTRATION["reasoning_stratum_auroc_min"]

if minority < min_n:
    print(f"  Still underpowered ({minority} < {min_n}). N_EXTRA was not enough; "
          f"draw more with load_fermat_extra_error_items(skip={SKIP + N_EXTRA}, ...) "
          "to extend further.")
elif error_stratum["auroc"] >= threshold and error_stratum["excludes_chance"]:
    print(f"  CONFIRMED: AUROC {error_stratum['auroc']:.3f} clears the registered "
          f"{threshold} threshold with adequate power ({minority} >= {min_n}).")
else:
    print(f"  NOT CONFIRMED: adequately powered ({minority} >= {min_n}) but AUROC "
          f"{error_stratum['auroc']:.3f} does not clear {threshold}, or its CI includes chance.")

print()
print("=" * 70)
print("CROSS-MODEL-FAMILY COMPARISON (this is the whole point of this notebook)")
print("=" * 70)
clean_stratum = out["strata"][False]
print(f"  LLaVA has_error=1 : {error_stratum['auroc']:.3f} "
      f"[{error_stratum['ci_low']:.3f}, {error_stratum['ci_high']:.3f}]  n_wrong={error_stratum['n_error']}")
print(f"  LLaVA clean stratum : {clean_stratum['auroc']:.3f} "
      f"[{clean_stratum['ci_low']:.3f}, {clean_stratum['ci_high']:.3f}]  n_wrong={clean_stratum['n_error']}")
print(f"  Qwen-7B has_error=1 (report S7.6): 0.801 [0.751, 0.846]")
print(f"  Qwen-7B clean stratum (report S7.2): 0.280 [0.200, 0.366]")
print("  If LLaVA's confirmed numbers land in the same ballpark and same")
print("  direction as Qwen's, the has_error=1 stratified effect and its")
print("  inversion on clean items generalize across model FAMILIES, not")
print("  just across sizes within one family.")


In [ ]:
# Save cell: Drive first, then repo + push. Distinctly named -- never
# overwrites the LLaVA reference results file.
import subprocess
from datetime import datetime, timezone
from getpass import getpass

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
csv_name = (f"grading_llava_stratum_powered_n{len(combined)}_"
            f"{'4bit_' if QUANTIZED else ''}{model_slug.lower()}_{timestamp}.csv")

drive_results = "/content/drive/MyDrive/uncertainty-math-vlm/results"
os.makedirs(drive_results, exist_ok=True)
combined.to_csv(f"{drive_results}/{csv_name}", index=False)
print(f"Backup written to {drive_results}/{csv_name}")

os.makedirs("repo/results", exist_ok=True)
csv_path = f"repo/results/{csv_name}"
combined.to_csv(csv_path, index=False)
print(f"Wrote {csv_path} ({len(combined)} rows)")

_REDACT = []


def git(*args):
    result = subprocess.run(["git", "-C", "repo", *args], capture_output=True, text=True)
    output = (result.stdout or "") + (result.stderr or "")
    for secret in _REDACT:
        if secret:
            output = output.replace(secret, "***")
    if result.returncode != 0 and output.strip():
        print(output.strip())
    return result


git("config", "user.email", "colab-pilot@localhost")
git("config", "user.name", "Colab Pilot Run")
git("add", f"results/{csv_name}")
commit = git("commit", "-m", f"Add LLaVA-NeXT stratum-powered grading results: {csv_name}")
if commit.returncode != 0:
    print("git commit failed (see above) -- CSV is safe on Drive.")

GH_PUSH_TOKEN = (globals().get("GH_TOKEN") or "").strip()
if not GH_PUSH_TOKEN:
    GH_PUSH_TOKEN = getpass("GitHub token (to push results), then press Enter: ").strip()
_REDACT.append(GH_PUSH_TOKEN)

if not GH_PUSH_TOKEN:
    print("No token given -- skipping push. CSV is saved on Drive and in repo/results/.")
else:
    push_url = REPO_URL.replace("https://", f"https://{GH_PUSH_TOKEN}@")
    if git("fetch", push_url, "main").returncode == 0:
        if git("rebase", "FETCH_HEAD").returncode != 0:
            git("rebase", "--abort")
            print("Rebase onto remote failed; attempting push anyway.")
    if git("push", push_url, "HEAD:main").returncode == 0:
        print("Pushed results.")
    else:
        print("Push failed (see above). The CSV is safe on Drive and in "
              "repo/results/ -- retry the push without re-running the model.")
